# ECON 1307 — Clase 7
## Indexacion, Filtrado, Duplicados y Limpieza

**Objetivo:** Aprende a seleccionar, filtrar y limpiar datos reales.

---

David Rodríguez 202114940
Manuel Garzón 202520480

In [1]:
import pandas as pd
import numpy as np

print('Librerias cargadas')


Librerias cargadas


## Paso 1: Cargar los datos

Ejecuta la siguiente celda. El dataset tiene 500+ empleados con:
- ID_empleado: identificador único de empleado
- Nombre_Apellido: nombre completo (identificador legible)
- Edad, Sexo, Sector, etc.

Nota: Hay DUPLICADOS intencionales y valores nulos (NaN) para que practiques limpieza.


In [2]:
# EJECUTA ESTA CELDA - Los datos ya estan listos
np.random.seed(42)

nombres = ['Juan', 'Maria', 'Carlos', 'Ana', 'Diego', 'Laura', 'Jose', 'Carmen', 
           'Miguel', 'Isabel', 'Antonio', 'Rosa', 'Francisco', 'Pedro', 'Andres',
           'Elena', 'Manuel', 'Lucia', 'Raul', 'Beatriz', 'Fernando', 'Gabriela',
           'Sergio', 'Monica', 'Ricardo', 'Sandra', 'Javier', 'Valentina', 'Luis',
           'Alejandra', 'Marco', 'Natalia', 'Pablo', 'Sofia', 'Gustavo', 'Camila',
           'Enrique', 'Mariana', 'Eduardo', 'Daniela', 'Felipe', 'Catalina', 'Hector',
           'Paulina', 'Ignacio', 'Roxana', 'Nicolas', 'Francisca']

apellidos = ['Garcia', 'Martinez', 'Lopez', 'Gonzalez', 'Rodriguez', 'Fernandez', 'Sanchez',
             'Perez', 'Torres', 'Ramirez', 'Flores', 'Rivera', 'Morales', 'Ortiz', 'Gutierrez',
             'Romero', 'Jimenez', 'Herrera', 'Soto', 'Espinoza', 'Vega', 'Rojas', 'Medina',
             'Mendoza', 'Salazar', 'Munoz', 'Vargas', 'Contreras', 'Diaz', 'Acosta', 'Carrillo',
             'Castro', 'Valencia', 'Bravo', 'Reyes', 'Ayala', 'Silva', 'Pacheco', 'Quezada',
             'Fuentes', 'Castillo', 'Cortes', 'Barrera', 'Figueroa', 'Campos', 'Avila', 'Montoya']

combinaciones = []
for i in range(500):
    nom = nombres[i % len(nombres)]
    ape = apellidos[i % len(apellidos)]
    combinaciones.append(nom + ' ' + ape)

combinaciones = list(set(combinaciones))
while len(combinaciones) < 500:
    for i in range(len(nombres)):
        for j in range(len(apellidos)):
            comb = nombres[i] + ' ' + apellidos[j]
            if comb not in combinaciones:
                combinaciones.append(comb)
                if len(combinaciones) >= 500:
                    break
        if len(combinaciones) >= 500:
            break

combinaciones = combinaciones[:500]

datos = {
    'ID_empleado': np.arange(1000, 1500),
    'Nombre_Apellido': combinaciones,
    'Edad': np.random.randint(18, 70, 500),
    'Sexo': np.random.choice(['M', 'F', 'Otro', np.nan], 500, p=[0.4, 0.4, 0.05, 0.15]),
    'Ingresos': np.random.choice(list(range(1500000, 8000000, 100000)) + [np.nan], 500),
    'Sector': np.random.choice(['Salud', 'Educacion', 'Tecnologia', 'Finanzas', 'Manufactura', np.nan], 500, p=[0.25, 0.2, 0.25, 0.15, 0.1, 0.05]),
    'Anos_Experiencia': np.random.choice(list(range(0, 40)) + [np.nan], 500),
    'Certificado': np.random.choice(['Si', 'No', np.nan], 500, p=[0.3, 0.55, 0.15]),
}

df = pd.DataFrame(datos)

# Agregar MUCHOS duplicados
duplicados_idx = np.random.choice(df.index, 30, replace=False)
df = pd.concat([df, df.loc[duplicados_idx]], ignore_index=True)

# Agregar NaNs
df.loc[np.random.choice(df.index, 30), 'Ingresos'] = np.nan
df.loc[np.random.choice(df.index, 20), 'Anos_Experiencia'] = np.nan

# Mezclar
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print('Dataset cargado:')
print('Filas:', len(df))
print('Columnas:', len(df.columns))


Dataset cargado:
Filas: 530
Columnas: 8


## Paso 2: Ver los datos


In [3]:
print('Primeras 10 filas:')
print(df.head(10))


Primeras 10 filas:
   ID_empleado     Nombre_Apellido  Edad  Sexo   Ingresos      Sector  \
0         1140    Catalina Barrera    54   nan  6700000.0       Salud   
1         1398  Francisca Martinez    59     M  6500000.0  Tecnologia   
2         1006     Gabriela Vargas    38     M  5700000.0    Finanzas   
3         1334      Antonio Romero    50     F  7300000.0       Salud   
4         1322      Carlos Morales    42     F  4200000.0         nan   
5         1082    Carlos Rodriguez    65     M  3300000.0   Educacion   
6         1225    Paulina Figueroa    66     F  2700000.0   Educacion   
7         1496         Laura Perez    42  Otro  6300000.0       Salud   
8         1248    Daniela Figueroa    22     M  3900000.0  Tecnologia   
9         1101        Roxana Lopez    18     F  4400000.0       Salud   

   Anos_Experiencia Certificado  
0              10.0          No  
1              31.0         nan  
2               NaN          Si  
3              25.0         nan  
4      

In [4]:
print('Información del dataset:')
df.info()


Información del dataset:
<class 'pandas.DataFrame'>
RangeIndex: 530 entries, 0 to 529
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID_empleado       530 non-null    int64  
 1   Nombre_Apellido   530 non-null    str    
 2   Edad              530 non-null    int32  
 3   Sexo              530 non-null    str    
 4   Ingresos          494 non-null    float64
 5   Sector            530 non-null    str    
 6   Anos_Experiencia  498 non-null    float64
 7   Certificado       530 non-null    str    
dtypes: float64(2), int32(1), int64(1), str(4)
memory usage: 31.2 KB


In [5]:
print('Valores nulos:')
print(df.isnull().sum())


Valores nulos:
ID_empleado          0
Nombre_Apellido      0
Edad                 0
Sexo                 0
Ingresos            36
Sector               0
Anos_Experiencia    32
Certificado          0
dtype: int64


---

## EJERCICIO 1: Ver datos basico

Muestra las primeras 5 filas del dataset.


In [27]:
# ESCRIBE AQUI:

df.head()

df.tail()

,ID_empleado,Nombre_Apellido,Edad,Sexo,Ingresos,Sector,Anos_Experiencia,Certificado
525,1071,Sofia Cortes,59,nan,3000000.0,Manufactura,20.0,No
526,1106,Sandra Vargas,18,M,NaN,Finanzas,38.0,No
527,1270,Beatriz Acosta,24,nan,5300000.0,Salud,14.0,Si
528,1435,Daniela Fuentes,63,F,4100000.0,Educacion,33.0,No
529,1102,Juan Sanchez,42,F,7300000.0,Finanzas,18.0,Si


---

## PARTE 1: Seleccionar filas y columnas

Aprenderemos a elegir exactamente qué datos queremos ver.


### Ejemplo 1: Ver solo una columna


In [7]:
# Ver solo los nombres (primeras 10)
print(df['Nombre_Apellido'].head(10))


0      Catalina Barrera
1    Francisca Martinez
2       Gabriela Vargas
3        Antonio Romero
4        Carlos Morales
5      Carlos Rodriguez
6      Paulina Figueroa
7           Laura Perez
8      Daniela Figueroa
9          Roxana Lopez
Name: Nombre_Apellido, dtype: str


### Ejemplo 2: Ver varias columnas


In [8]:
# Ver: Nombre_Apellido, Edad, Sector (primeras 10)
print(df[['Nombre_Apellido', 'Edad', 'Sector']].head(10))


      Nombre_Apellido  Edad      Sector
0    Catalina Barrera    54       Salud
1  Francisca Martinez    59  Tecnologia
2     Gabriela Vargas    38    Finanzas
3      Antonio Romero    50       Salud
4      Carlos Morales    42         nan
5    Carlos Rodriguez    65   Educacion
6    Paulina Figueroa    66   Educacion
7         Laura Perez    42       Salud
8    Daniela Figueroa    22  Tecnologia
9        Roxana Lopez    18       Salud


### Ejemplo 3: Ver primeras filas y columnas específicas


In [9]:
# Ver primeras 5 filas, columnas: ID_empleado, Nombre_Apellido, Edad
print(df.loc[:4, ['ID_empleado', 'Nombre_Apellido', 'Edad']])


   ID_empleado     Nombre_Apellido  Edad
0         1140    Catalina Barrera    54
1         1398  Francisca Martinez    59
2         1006     Gabriela Vargas    38
3         1334      Antonio Romero    50
4         1322      Carlos Morales    42


### EJERCICIO 2

Muestra:
- Las columnas: Nombre_Apellido, Ingresos, Certificado
- Solo las primeras 5 filas


In [28]:
# ESCRIBE AQUI:

print(df.loc[:9, ['Ingresos', 'Nombre_Apellido', 'Certificado']])


    Ingresos     Nombre_Apellido Certificado
0  6700000.0    Catalina Barrera          No
1  6500000.0  Francisca Martinez         nan
2  5700000.0     Gabriela Vargas          Si
3  7300000.0      Antonio Romero         nan
4  4200000.0      Carlos Morales          Si
5  3300000.0    Carlos Rodriguez          No
6  2700000.0    Paulina Figueroa          No
7  6300000.0         Laura Perez          No
8  3900000.0    Daniela Figueroa          No
9  4400000.0        Roxana Lopez          No


---

## PARTE 2: Filtrado (encontrar datos especificos)

Ahora aprenderemos a encontrar solo lo que nos interesa.


### Ejemplo 4: Personas mayores de 50 años


In [11]:
mayores = df[df['Edad'] > 50]
print('Personas mayores de 50 años:', len(mayores))
print(mayores[['Nombre_Apellido', 'Edad', 'Sector']].head(10))


Personas mayores de 50 años: 198
       Nombre_Apellido  Edad      Sector
0     Catalina Barrera    54       Salud
1   Francisca Martinez    59  Tecnologia
5     Carlos Rodriguez    65   Educacion
6     Paulina Figueroa    66   Educacion
11     Roxana Martinez    53    Finanzas
12       Gustavo Silva    51   Educacion
13           Ana Perez    54       Salud
14         Diego Perez    57    Finanzas
15       Felipe Garcia    69  Tecnologia
17      Gabriela Rojas    56  Tecnologia


### Ejemplo 5: Personas en el sector Tecnologia


In [12]:
tech = df[df['Sector'] == 'Tecnologia']
print('Personas en Tecnologia:', len(tech))
print(tech[['Nombre_Apellido', 'Sector']].head(10))


Personas en Tecnologia: 145
       Nombre_Apellido      Sector
1   Francisca Martinez  Tecnologia
8     Daniela Figueroa  Tecnologia
15       Felipe Garcia  Tecnologia
16        Sandra Munoz  Tecnologia
17      Gabriela Rojas  Tecnologia
22      Paulina Garcia  Tecnologia
23      Pablo Valencia  Tecnologia
28    Francisca Garcia  Tecnologia
32        Maria Rivera  Tecnologia
41       Rosa Espinoza  Tecnologia


### Ejemplo 6: Personas con certificado


In [13]:
certificados = df[df['Certificado'] == 'Si']
print('Personas con certificado:', len(certificados))


Personas con certificado: 160


### EJERCICIO 3

Encuentra personas de 25 años y NO tienen certificado.
¿Cuantas hay?


In [30]:
# ESCRIBE AQUI:

condicion = (df['Edad'] == 25) & (df['Certificado'] == 'No')
print('Personas de 25 años sin certificado:', len(df[condicion]))

Personas de 25 años sin certificado: 7


### EJERCICIO 4

Encuentra personas en el sector 'Salud'.
¿Cuantas hay?


In [15]:
# ESCRIBE AQUI:


---

## PARTE 3: TRATAMIENTO DE DUPLICADOS

A veces los datos tienen filas repetidas. Necesitamos:
1. Detectarlas
2. Entender si son duplicados por qué columnas
3. Decidir qué hacer


### Ejemplo 7: Contar duplicados (fila completa)


In [16]:
print('Total de filas:', len(df))
print('Filas duplicadas (fila completa):', df.duplicated().sum())
print('Filas unicas:', len(df) - df.duplicated().sum())


Total de filas: 530
Filas duplicadas (fila completa): 24
Filas unicas: 506


### Ejemplo 8: Ver filas duplicadas


In [17]:
# Mostrar filas que aparecen mas de una vez
duplicadas = df[df.duplicated(keep=False)].sort_values('ID_empleado')
print('Primeras filas duplicadas (ordenadas por ID_empleado):')
print(duplicadas.head(10))


Primeras filas duplicadas (ordenadas por ID_empleado):
     ID_empleado  Nombre_Apellido  Edad  Sexo   Ingresos       Sector  \
84          1020   Carmen Jimenez    61     M  6300000.0        Salud   
524         1020   Carmen Jimenez    61     M  6300000.0        Salud   
370         1041  Gustavo Pacheco    54     M  1500000.0  Manufactura   
435         1041  Gustavo Pacheco    54     M  1500000.0  Manufactura   
260         1059      Juan Flores    61   nan  2100000.0  Manufactura   
491         1059      Juan Flores    61   nan  2100000.0  Manufactura   
348         1065  Roxana Martinez    53  Otro  6500000.0     Finanzas   
11          1065  Roxana Martinez    53  Otro  6500000.0     Finanzas   
238         1074     Manuel Rojas    35     F  2800000.0  Manufactura   
478         1074     Manuel Rojas    35     F  2800000.0  Manufactura   

     Anos_Experiencia Certificado  
84                7.0          No  
524               7.0          No  
370              31.0          No

### Ejemplo 9: Detectar duplicados por Nombre_Apellido

¿Y si queremos ver si la MISMA PERSONA aparece mas de una vez?


In [18]:
# Duplicados solo por Nombre_Apellido
print('Filas duplicadas por Nombre_Apellido:', df.duplicated(subset=['Nombre_Apellido'], keep=False).sum())

# Ver ejemplos
dup_por_nombre = df[df.duplicated(subset=['Nombre_Apellido'], keep=False)].sort_values('Nombre_Apellido')
print('\nEjemplos de personas que aparecen mas de una vez:')
print(dup_por_nombre[['ID_empleado', 'Nombre_Apellido', 'Edad', 'Sector']].head(10))


Filas duplicadas por Nombre_Apellido: 60

Ejemplos de personas que aparecen mas de una vez:
     ID_empleado   Nombre_Apellido  Edad      Sector
349         1283   Antonio Jimenez    40         nan
337         1283   Antonio Jimenez    40         nan
524         1020    Carmen Jimenez    61       Salud
84          1020    Carmen Jimenez    61       Salud
210         1248  Daniela Figueroa    22  Tecnologia
8           1248  Daniela Figueroa    22  Tecnologia
434         1275   Daniela Montoya    56       Salud
272         1275   Daniela Montoya    56       Salud
61          1145        Elena Soto    22  Tecnologia
225         1145        Elena Soto    22  Tecnologia


### Ejemplo 10: Eliminar duplicados (fila completa)


In [19]:
print('Antes de eliminar duplicados:', len(df), 'filas')
df_sin_dup = df.drop_duplicates()
print('Despues de eliminar duplicados:', len(df_sin_dup), 'filas')
print('Se eliminaron:', len(df) - len(df_sin_dup), 'filas')


Antes de eliminar duplicados: 530 filas
Despues de eliminar duplicados: 506 filas
Se eliminaron: 24 filas


### Ejemplo 11: Eliminar duplicados solo por una columna


In [20]:
# Si eliminamos duplicados por Nombre_Apellido, mantenemos la PRIMERA ocurrencia
print('Antes:', len(df), 'filas')
df_sin_dup_nombre = df.drop_duplicates(subset=['Nombre_Apellido'], keep='first')
print('Despues (sin duplicados por Nombre_Apellido):', len(df_sin_dup_nombre), 'filas')


Antes: 530 filas
Despues (sin duplicados por Nombre_Apellido): 500 filas


### EJERCICIO 5

¿Cuantas filas quedan despues de eliminar duplicados (fila completa)?


In [21]:
# ESCRIBE AQUI:


### EJERCICIO 6

¿Cuantas filas quedan si eliminamos duplicados SOLO por Nombre_Apellido?


In [22]:
# ESCRIBE AQUI:


---

## PARTE 4: Limpieza (llenar valores nulos)

Los datos a veces tienen valores faltantes (NaN = No es un numero).
Podemos eliminarlos o rellenarlos.


### Ejemplo 12: Contar valores nulos


In [23]:
print('Valores nulos en cada columna:')
print(df.isnull().sum())


Valores nulos en cada columna:
ID_empleado          0
Nombre_Apellido      0
Edad                 0
Sexo                 0
Ingresos            36
Sector               0
Anos_Experiencia    32
Certificado          0
dtype: int64


### Ejemplo 13: Eliminar filas con valores nulos


In [24]:
print('Antes de eliminar NaN:', len(df), 'filas')
df_sin_nan = df.dropna()
print('Despues de eliminar NaN:', len(df_sin_nan), 'filas')


Antes de eliminar NaN: 530 filas
Despues de eliminar NaN: 465 filas


### Ejemplo 14: Rellenar valores nulos con un valor


In [25]:
df_relleno = df.copy()
df_relleno['Certificado'] = df_relleno['Certificado'].fillna('No')
df_relleno['Sexo'] = df_relleno['Sexo'].fillna('Sin dato')

print('Valores nulos DESPUES de rellenar:')
print(df_relleno.isnull().sum())


Valores nulos DESPUES de rellenar:
ID_empleado          0
Nombre_Apellido      0
Edad                 0
Sexo                 0
Ingresos            36
Sector               0
Anos_Experiencia    32
Certificado          0
dtype: int64


### EJERCICIO 7

Rellena los valores nulos en la columna 'Anos_Experiencia' con 0.

¿Cuantos valores nulos hay en esa columna ANTES de rellenar?

¿Cuantos hay DESPUES?


In [26]:
# ESCRIBE AQUI:


---

## TAREA

**Objetivo:** Practicar lo que aprendiste.

### Instrucciones

Usa el dataset original (con duplicados y NaN) para responder:

### Filtrado basico 

a) ¿Cuantas personas hay en el sector 'Salud'?

b) ¿Cuantas personas tienen edad menor a 30?

c) ¿Cuantas personas tienen certificado 'Si'?

### Duplicados 

a) ¿Cuantas filas duplicadas hay en total (fila completa)?

b) ¿Cuantas personas diferentes (Nombre_Apellido) tienen registros duplicados?

c) Si eliminas duplicados por Nombre_Apellido, ¿cuantas filas quedan?

###  Limpieza 

a) ¿Cuantos valores nulos hay en la columna 'Ingresos'?

b) ¿Cuantos valores nulos hay en la columna 'Anos_Experiencia'?

c) Rellena 'Anos_Experiencia' con 0. ¿Hay mas valores nulos en esa columna?


In [34]:
# Filtrado Básico
# a) Cuántas personas hay en el sector salud?
salud = df[df['Sector'] == 'Salud']
print('Personas en el sector salud:', len(salud))

Personas en el sector salud: 136


In [35]:
# b) Cuántas personas tienen menos de 30 años?
menores_30 = df[df['Edad'] < 30]
print('Personas menores de 30 años:', len(menores_30))  

Personas menores de 30 años: 116


In [36]:
# c) Cuántas personas tienen certificado sí?
certificado_si = df[df['Certificado'] == 'Si']
print('Personas con certificado sí:', len(certificado_si))

Personas con certificado sí: 160


In [37]:
# Duplicados
# a) Cuántas filas duplicadas hay en el dataset?
print('Filas duplicadas (fila completa):', df.duplicated().sum())

Filas duplicadas (fila completa): 24


In [ ]:
# b) ¿Cuantas personas diferentes (Nombre_Apellido) tienen registros duplicados?
duplicados = df[df.duplicated(subset=['Nombre_Apellido'], keep=False)]
print('Personas con registros duplicados:', len(duplicados))
print(duplicados)

Personas con registros duplicados: 60
     ID_empleado     Nombre_Apellido  Edad  Sexo   Ingresos       Sector  \
8           1248    Daniela Figueroa    22     M  3900000.0   Tecnologia   
11          1065     Roxana Martinez    53  Otro  6500000.0     Finanzas   
18          1257         Sofia Reyes    26   nan  7600000.0     Finanzas   
19          1171         Jose Torres    53     M  1700000.0  Manufactura   
20          1478        Felipe Lopez    59     F  6000000.0     Finanzas   
24          1335   Francisca Ramirez    21     F  7300000.0    Educacion   
25          1371      Juan Rodriguez    52     F  7700000.0  Manufactura   
32          1444        Maria Rivera    19     M  1700000.0   Tecnologia   
44          1491       Isabel Rivera    18     F        NaN   Tecnologia   
45          1249      Lucia Espinoza    69     F  1900000.0          nan   
50          1479        Jose Jimenez    47     F  4600000.0        Salud   
61          1145          Elena Soto    22   nan  

In [40]:
# c) c) Si eliminas duplicados por Nombre_Apellido, ¿cuantas filas quedan?
df_sin_duplicados = df.drop_duplicates(subset=['Nombre_Apellido'])
print('Filas después de eliminar duplicados por Nombre_Apellido:', len(df_sin_duplicados))

Filas después de eliminar duplicados por Nombre_Apellido: 500


In [44]:
# Limpieza

# a) ¿Cuantos valores nulos hay en la columna 'Ingresos'?
print('Valores nulos en Ingresos:', df['Ingresos'].isnull().sum())

# b) ¿Cuantos valores nulos hay en la columna 'Anos_Experiencia'?
print('Valores nulos en Anos_Experiencia:', df['Anos_Experiencia'].isnull().sum())

# c) Rellena 'Anos_Experiencia' con 0. ¿Hay mas valores nulos en esa columna?
df_relleno_experiencia = df.copy()
df_relleno_experiencia['Anos_Experiencia'] = df_relleno_experiencia['Anos_Experiencia'].fillna(0)
print('Valores nulos en Anos_Experiencia después de rellenar con 0:', df_relleno_experiencia['Anos_Experiencia'].isnull().sum())

Valores nulos en Ingresos: 36
Valores nulos en Anos_Experiencia: 32
Valores nulos en Anos_Experiencia después de rellenar con 0: 0
